# RAG OWASP: implementação e avaliação

**Entrega:** notebook executado + relatório de até duas páginas. Este notebook cobre P1–P7 do roadmap.

P1–P3 reutilizam os scripts do grupo. P4–P7 acrescentam geração, comparação, avaliação manual e relatório.
Usamos um modelo **gratuito e pré-treinado**, sem API paga nem fine-tuning. O corpus OWASP é público e em inglês;
perguntas e respostas permanecem em inglês para reduzir diferenças de idioma entre os retrievers.

A atividade pede três perguntas com/sem recuperação (seis respostas) e dez respostas avaliadas.
Adotamos **cinco perguntas em ambas as condições**, mantendo as três primeiras como comparação mínima.
Confirme essa interpretação com o professor antes da entrega.

**No Colab:** selecione Ambiente de execução > Alterar tipo > GPU. Abra este notebook e envie
`owasp-rag-colab.zip` quando solicitado. O ZIP contém o código modificado, não apenas o clone original.


In [ ]:
from pathlib import Path
import sys
import subprocess
import zipfile

IN_COLAB = "google.colab" in sys.modules
candidates = [Path.cwd(), Path.cwd().parent, Path("/content/owasp-rag")]
PROJECT_DIR = next((p for p in candidates if (p / "src/run_experiment.py").is_file()), None)
if PROJECT_DIR is None and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Envie exatamente um ZIP contendo o projeto modificado.")
    destination = Path("/content")
    with zipfile.ZipFile(archives[0]) as archive:
        for info in archive.infolist():
            target = (destination / info.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError("Caminho inseguro no ZIP.")
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Links simbólicos não são aceitos no ZIP.")
        archive.extractall(destination)
    PROJECT_DIR = destination / "owasp-rag"
if PROJECT_DIR is None or not (PROJECT_DIR / "src/run_experiment.py").is_file():
    raise FileNotFoundError("Execute dentro de owasp-rag ou envie o ZIP atualizado no Colab.")
PROJECT_DIR = PROJECT_DIR.resolve()
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(PROJECT_DIR / "src"))
print("Projeto:", PROJECT_DIR)


## P1: corpus e chunks existentes

Baixamos os documentos apenas quando o corpus processado não existe. Os documentos selecionados,
textos completos, URLs e IDs são preservados. Chunks: 200 palavras, sobreposição: 40.
Se modificar o corpus, use outro diretório de experimento; não misture notas antigas com novas respostas.


In [ ]:
import json
from config import CHUNKS_DIR, EXTRACTED_DIR
from retrieve_tfidf import load_chunks

if not (CHUNKS_DIR / "chunks.json").exists():
    from download_docs import main as download_documents
    from build_dataset import main as build_dataset
    from chunk_documents import main as chunk_documents
    download_documents()
    build_dataset()
    chunk_documents()

chunks = load_chunks()
corpus = json.loads((EXTRACTED_DIR / "documents.json").read_text(encoding="utf-8"))
chunk_stats = json.loads((CHUNKS_DIR / "stats.json").read_text(encoding="utf-8"))
assert 20 <= corpus["document_count"] <= 30
assert len({c["chunk_id"] for c in chunks}) == len(chunks)
print(json.dumps(chunk_stats, ensure_ascii=False, indent=2))


## Configuração do experimento

`Qwen/Qwen2.5-1.5B-Instruct` cabe na GPU gratuita do Colab. O script escolhe CUDA quando disponível;
em CPU, usa float32 e pode demorar mais. O download inicial ocupa alguns GB. O modelo é público;
não é necessário token Hugging Face. RAG usa evidência no prompt; **não treina novamente o LLM**.

Para uma nova execução no Colab, mantenha `OUTPUT_DIR` abaixo em `experiment_colab`.
Os resultados locais fornecidos ficam em `output/experiment`. Para inspecioná-los sem baixar modelos,
altere `OUTPUT_DIR` para `PROJECT_DIR / "output" / "experiment"`.
Um experimento completo só é reutilizado quando corpus, perguntas, modelo e parâmetros coincidem.


In [ ]:
import torch
from run_experiment import load_questions

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
RETRIEVER = "dense"  # também aceita "tfidf"
K = 3
MAX_NEW_TOKENS = 256
MAX_INPUT_TOKENS = 4096
OUTPUT_DIR = PROJECT_DIR / "output" / ("experiment_colab" if IN_COLAB else "experiment")
questions = load_questions()
assert len(questions) == 5, "Para a entrega, use cinco perguntas e dez respostas."
missing_sources = {doc_id for q in questions for doc_id in q.get("expected_doc_ids", [])} - {
    chunk["doc_id"] for chunk in chunks
}
if missing_sources:
    raise ValueError(f"Documentos de referência ausentes do corpus: {missing_sources}")
print("Dispositivo:", "CUDA" if torch.cuda.is_available() else "CPU")
print("Modelo:", MODEL_NAME)
print("Perguntas:", len(questions), "| Respostas previstas:", 2 * len(questions))


## P2–P5: rankings e respostas com/sem recuperação

Os dois retrievers recebem exatamente os mesmos chunks e perguntas. A geração RAG usa apenas o
retriever escolhido acima; o outro serve como comparação independente da recuperação.

As referências de correção **não entram no prompt**. O mesmo LLM, instruções de sistema e decodificação
sem amostragem são usados nas duas condições. A diferença é a presença dos chunks.
Antes da primeira geração, salvamos ambos os rankings. Cada resposta registra prompt exato, chunks
completos, scores, modelo, revisão, k, parâmetros, tokens, limite de saída e checagem dos IDs citados.


In [ ]:
from run_experiment import compare_retrievers, load_experiment, run_experiment
from generate_answers import LocalLLM

if (OUTPUT_DIR / "experiment.json").exists():
    runs, rankings = load_experiment(OUTPUT_DIR, questions, MODEL_NAME, RETRIEVER, K,
                                    MAX_NEW_TOKENS, MAX_INPUT_TOKENS)
    print("Resultados completos reutilizados; fingerprints e parâmetros conferidos.")
else:
    rankings = compare_retrievers(questions, K)
    generator = LocalLLM(MODEL_NAME, MAX_NEW_TOKENS, MAX_INPUT_TOKENS)
    runs = run_experiment(questions, generator, RETRIEVER, K, OUTPUT_DIR, rankings)
    del generator
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
assert len(runs) == 10
print("Respostas registradas:", len(runs))


In [ ]:
from html import escape
from IPython.display import HTML, Markdown, display

for question in questions:
    qid = question["question_id"]
    display(Markdown(f"### {qid}: {question['question']}"))
    for ranking in [r for r in rankings if r["question_id"] == qid]:
        rows = "".join(
            f"<tr><td>{c['rank']}</td><td>{escape(c['chunk_id'])}</td>"
            f"<td>{c['score']:.4f}</td><td>{escape(c['title'])}</td></tr>"
            for c in ranking["chunks"]
        )
        display(HTML(f"<strong>{escape(ranking['retriever'])}, k={ranking['k']}</strong>"
                     "<table><tr><th>Rank</th><th>Chunk</th><th>Cosseno</th><th>Documento</th></tr>"
                     + rows + "</table>"))
        for chunk in ranking["chunks"]:
            display(HTML("<details><summary>" + escape(chunk["chunk_id"]) + "</summary><p>"
                         + escape(chunk["source_url"]) + "</p><pre style='white-space:pre-wrap'>"
                         + escape(chunk["text"]) + "</pre></details>"))
    for run in [r for r in runs if r["question_id"] == qid]:
        display(Markdown(f"**{run['condition']}**"))
        display(HTML("<pre style='white-space:pre-wrap'>" + escape(run["answer"]) + "</pre>"))
        print("IDs citados:", run["citation_check"]["cited_chunk_ids"],
              "| IDs desconhecidos:", run["citation_check"]["unknown_citation_ids"],
              "| Saída atingiu limite:", run["hit_output_limit"])
        display(HTML("<details><summary>Prompt exato</summary><pre style='white-space:pre-wrap'>"
                     + escape(run["prompt"]) + "</pre></details>"))


## P6: avaliação manual separada

**Primeiro, recuperação:** abra `retrieval_grades.csv`. Para cada pergunta e retriever, marque
`evidence_found=yes` apenas se os top-k contêm **todas** as informações necessárias. Indique os IDs
que sustentam essa decisão em `evidence_chunk_ids`, separados por `;`, e justifique em `notes`.
Um documento esperado no ranking não prova que o trecho recuperado contém a evidência.

**Depois, dez respostas:** abra `answer_grades.csv` e preencha:

- `correct`: `yes` se os fatos estão corretos e todas as partes foram respondidas; caso contrário, `no`.
- `supported`: `yes` se todas as afirmações factuais são sustentadas pelo contexto; caso contrário, `no`.
- `citation_correct`: `yes` se as citações apontam para os trechos que sustentam as afirmações; caso contrário, `no`.
- `evidence_chunk_ids`: IDs do contexto efetivamente usados na revisão, separados por `;`.
- `notes`: justificativa humana, incluindo fatos ausentes, incorretos ou sem suporte.

Para respostas **sem recuperação**, `supported` e `citation_correct` ficam `na` (não aplicável).
Não trate falta de contexto no baseline como falha de recuperação. IDs válidos sozinhos não provam suporte.
A checagem automática nunca preenche notas de correção. Referências abaixo são guias de revisão;
confira o texto original do corpus, especialmente após atualizações da OWASP.


In [ ]:
from evaluate_answers import write_evaluation_templates

write_evaluation_templates(runs, rankings, OUTPUT_DIR)  # preserva avaliações já preenchidas
for question in questions:
    display(Markdown(f"**Referência de {question['question_id']}:** {question['reference_answer']}"))
    for doc in corpus["documents"]:
        if doc["doc_id"] in question.get("expected_doc_ids", []):
            display(HTML("<details><summary>Documento completo de referência: " + escape(doc["title"])
                         + "</summary><pre style='white-space:pre-wrap'>" + escape(doc["text"])
                         + "</pre></details>"))
print("Arquivos para revisão:")
for name in ["retrieval_grades.csv", "answer_grades.csv", "review.json"]:
    print(OUTPUT_DIR / name)


No Colab, baixe os três arquivos, preencha-os e reenvie-os na célula abaixo.
Também pode editá-los no painel Arquivos. Localmente, edite-os diretamente.
Para campos CSV com vírgulas, mantenha as aspas; prefira editor de planilhas e salve em UTF-8.


In [ ]:
UPLOAD_MANUAL_REVIEWS = False  # mude para True ao reenviar avaliações preenchidas no Colab
if UPLOAD_MANUAL_REVIEWS:
    if not IN_COLAB:
        raise ValueError("Localmente, edite os arquivos diretamente.")
    from google.colab import files
    uploaded = files.upload()
    allowed = {"retrieval_grades.csv", "answer_grades.csv", "review.json"}
    if set(uploaded) != allowed:
        raise ValueError("Reenvie exatamente os três arquivos de avaliação.")
    for name, content in uploaded.items():
        (OUTPUT_DIR / name).write_bytes(content)
    print("Avaliações recebidas; serão validadas na próxima célula.")


## P7: dois erros, risco e relatório

Em `review.json`, preencha `failure_run_ids` com dois IDs com **falhas realmente avaliadas à mão**,
por exemplo `q2_with_retrieval`. Também são falhas RAG as respostas avaliadas como
`supported=no` ou `citation_correct=no`, conforme o roadmap. O código classifica o erro a partir da avaliação humana dos top-k:
evidência ausente = recuperação; evidência presente = geração. Um erro sem recuperação é de geração
ou conhecimento do baseline. Justifique cada erro em `answer_grades.csv` com exemplos concretos.

Se houver menos de dois erros, reformule as perguntas e gere **outro experimento**; não invente falhas.
O risco sugerido é manipulação por instruções dentro de documentos. Revise a mitigação em `review.json`.
O prompt delimita os dados e manda ignorar essas instruções, mas isso não garante proteção completa.

Enquanto faltarem avaliações, o PDF será identificado como **rascunho**. O relatório final só é gerado
com dez notas completas, avaliações dos dois retrievers e análise de dois erros. O exportador verifica
que o PDF tem exatamente duas páginas e pede notas menores se o conteúdo ultrapassar esse limite.


In [ ]:
from evaluate_answers import read_evaluation
from build_report import build_report

review = read_evaluation(OUTPUT_DIR)
print("Campos manuais pendentes:", len(review["pending"]))
for item in review["pending"]:
    print("-", item)
report_path = build_report(OUTPUT_DIR, draft=bool(review["pending"]))
print("Relatório:", report_path)
print("Status:", "RASCUNHO: completar revisão humana" if review["pending"] else "PRONTO para revisão final")


## Entrega

1. Preencha as avaliações e escolha os dois erros; execute novamente as células de avaliação e relatório.
2. Confira visualmente `report.pdf`: máximo duas páginas, notas e respostas aos itens 6 e 7.
3. No Colab, use **Arquivo > Fazer download > Fazer download de .ipynb** para salvar o notebook com outputs.
4. Entregue esse notebook e `report.pdf`. Um rascunho não substitui o relatório final.

A célula abaixo é opcional e baixa o PDF final no Colab. Ela recusa baixar um rascunho como entrega.


In [ ]:
DOWNLOAD_FINAL_REPORT = False
if DOWNLOAD_FINAL_REPORT:
    if review["pending"]:
        raise ValueError("Complete a avaliação manual antes de baixar a entrega final.")
    if IN_COLAB:
        from google.colab import files
        files.download(str(report_path))
    else:
        print("PDF final disponível em:", report_path)
